# pass@k Run Analysis

Loads samples.csv and pass_at_k.csv from a benchmark run. Supports model and temperature sweeps.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import plotly.express as px

RUN_NAME = None  # e.g. "tinker_family_pairs_t07_k1_5_10_20"
RUN_NAMES = [
    "tinker_qwen32b_t05_k1_20",
    "tinker_qwen8b_t05_k1_20",
    "tinker_llama31_8b_t05_k1_20",
    "tinker_llama33_70b_t05_k1_20",
    "tinker_finetuned_m2_qwen35_4b_t05_k1_20"
]

candidates = [Path("results/runs"), Path("../results/runs")]
RUN_ROOT = next((p.resolve() for p in candidates if p.exists()), candidates[0].resolve())
PROJECT_ROOT = RUN_ROOT.parent.parent
EVAL_PIPELINE = PROJECT_ROOT / "eval-pipeline"
if str(EVAL_PIPELINE) not in sys.path:
    sys.path.insert(0, str(EVAL_PIPELINE))

if RUN_NAMES:
    run_dirs = [RUN_ROOT / name for name in RUN_NAMES]
elif RUN_NAME:
    run_dirs = [RUN_ROOT / RUN_NAME]
else:
    runs = sorted([p for p in RUN_ROOT.glob("*") if p.is_dir()])
    run_dirs = [runs[-1]] if runs else []

assert run_dirs, f"No run found under {RUN_ROOT}"
for run_dir in run_dirs:
    assert run_dir.exists(), f"Missing run folder: {run_dir}"

def rebuild_csvs_from_completed_model_jsons(run):
    from csv_exports import write_run_csvs
    from eval import _file_key, generate_summary

    config_path = run / "run_config.json"
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    provider = config.get("provider", "tinker")
    models = config.get("models", [])
    pass_k = config.get("pass_k", [1, 5, 10, 15, 20])

    all_results = {}
    for model in models:
        result_path = run / f"{_file_key(provider, model)}_results.json"
        if result_path.exists():
            all_results[f"{provider}/{model}"] = json.loads(result_path.read_text())

    if not all_results:
        for result_path in sorted(run.glob("*_results.json")):
            all_results[result_path.stem.removesuffix("_results")] = json.loads(result_path.read_text())

    assert all_results, f"No completed *_results.json files found in {run}"
    summary = generate_summary(all_results, execute_mode=True, metric_names=["pass_at_k", "accuracy"], pass_k=pass_k)
    (run / "summary.json").write_text(json.dumps(summary, indent=2))
    write_run_csvs(run, all_results, summary)
    return list(all_results)

for run in run_dirs:
    pass_at_k_csv = run / "pass_at_k.csv"
    samples_csv = run / "samples.csv"
    if not pass_at_k_csv.exists() or not samples_csv.exists():
        rebuilt_models = rebuild_csvs_from_completed_model_jsons(run)
        print(f"Rebuilt {run.name} from {len(rebuilt_models)} completed model result file(s).")
    assert pass_at_k_csv.exists(), f"Missing pass@k CSV: {pass_at_k_csv}"
    assert samples_csv.exists(), f"Missing samples CSV: {samples_csv}"

run_dirs


In [ ]:
sample_frames = []
pass_at_k_frames = []
for run in run_dirs:
    sample_frame = pd.read_csv(run / "samples.csv")
    sample_frame.insert(0, "run_name", run.name)
    sample_frames.append(sample_frame)

    pass_frame = pd.read_csv(run / "pass_at_k.csv")
    pass_frame.insert(0, "run_name", run.name)
    pass_at_k_frames.append(pass_frame)

samples = pd.concat(sample_frames, ignore_index=True)
pass_at_k = pd.concat(pass_at_k_frames, ignore_index=True)
metric_cols = [c for c in pass_at_k.columns if c.startswith("pass@")]

aggregate = pass_at_k[pass_at_k["question_id"] == "__aggregate__"].copy()
aggregate


In [ ]:
MAX_LEGEND_LABEL_CHARS = 34
EXTRAPOLATE_K_MAX = 100

id_vars = [c for c in ["run_name", "model", "temperature"] if c in aggregate.columns]
long = aggregate.melt(
    id_vars=id_vars,
    value_vars=metric_cols,
    var_name="metric",
    value_name="pass@k",
)
long["k"] = pd.to_numeric(long["metric"].str.extract(r"pass@(\d+)")[0], errors="coerce")
long = long.dropna(subset=["k"]).copy()
long["k"] = long["k"].astype(int)
long["temperature_label"] = "T=" + long["temperature"].astype(str)

if "run_name" not in long.columns:
    long["run_name"] = long["model"].astype(str) + "_t" + long["temperature"].astype(str)

def truncate_label(value, max_chars=MAX_LEGEND_LABEL_CHARS):
    value = str(value)
    return value if len(value) <= max_chars else value[: max_chars - 1] + "..."

long["run_label"] = long["run_name"].map(truncate_label)
long = long.sort_values(["run_name", "model", "temperature", "k"])

run_order = list(dict.fromkeys(long["run_name"].astype(str)))
marker_symbols = ["circle", "square", "diamond", "cross", "x", "triangle-up", "triangle-down", "star"]
run_marker = {run: marker_symbols[i % len(marker_symbols)] for i, run in enumerate(run_order)}
run_label_marker = {truncate_label(run): run_marker[run] for run in run_order}

fig = px.line(
    long,
    x="k",
    y="pass@k",
    color="run_label",
    symbol="run_label",
    symbol_map=run_label_marker,
    markers=True,
    title="Pass@k by Run",
    labels={"k": "k", "pass@k": "pass@k", "run_label": "Run"},
    custom_data=["run_name", "model", "temperature"],
)
fig.update_traces(
    line_shape="spline",
    hovertemplate=(
        "run=%{customdata[0]}<br>"
        "model=%{customdata[1]}<br>"
        "temperature=%{customdata[2]}<br>"
        "k=%{x}<br>pass@k=%{y:.4f}<extra></extra>"
    ),
)
fig.update_layout(
    template="plotly_white",
    xaxis=dict(dtick=1),
    yaxis=dict(range=[0, 1]),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.22,
        xanchor="left",
        x=0,
        tracegroupgap=4,
    ),
    margin=dict(r=30, b=130),
)
fig.show()


## Linear-regression pass@k extrapolation

This fits the model from the note above to each aggregate model/temperature curve:

$$-\log(\widehat{\mathrm{pass@}k}) \sim a\log(k) + c,$$

so the displayed fitted curve is

$$\widehat{\mathrm{pass@}k}_\mathrm{fit} = \exp(-(a\log(k)+c)).$$

Only non-saturated observations with `0 < pass@k < 1` are used for the least-squares fit; all observed aggregate points are still plotted.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.colors import qualitative

fit_source = long.copy()
fit_source = fit_source.sort_values(["run_name", "model", "temperature", "k"])

fit_rows = []
fit_curves = []
group_cols = ["run_name", "run_label", "model", "temperature"]
for (run_name, run_label, model, temperature), group in fit_source.groupby(group_cols, dropna=False):
    observed = group[["k", "pass@k"]].dropna().copy()
    fit_data = observed[(observed["pass@k"] > 0) & (observed["pass@k"] < 1)].copy()
    if len(fit_data) < 2:
        continue

    x = np.log(fit_data["k"].to_numpy(dtype=float))
    y = -np.log(fit_data["pass@k"].to_numpy(dtype=float))
    a, c = np.polyfit(x, y, deg=1)

    k_min = int(observed["k"].min())
    observed_k_max = int(observed["k"].max())
    extrapolate_k_max = max(EXTRAPOLATE_K_MAX, observed_k_max)
    k_grid = np.arange(k_min, extrapolate_k_max + 1)
    fitted_pass = np.exp(-(a * np.log(k_grid) + c))
    fitted_pass = np.clip(fitted_pass, 0, 1)

    fit_rows.append({
        "run_name": run_name,
        "run_label": run_label,
        "model": model,
        "temperature": temperature,
        "a": a,
        "c": c,
        "C = exp(-c)": np.exp(-c),
        "observed_k_max": observed_k_max,
        "extrapolated_k_max": extrapolate_k_max,
        "fit_points": len(fit_data),
    })
    fit_curves.append(pd.DataFrame({
        "run_name": run_name,
        "run_label": run_label,
        "model": model,
        "temperature": temperature,
        "k": k_grid,
        "fitted_pass@k": fitted_pass,
        "is_extrapolated": k_grid > observed_k_max,
    }))

linear_fit_params = pd.DataFrame(fit_rows)
linear_fit_curves = pd.concat(fit_curves, ignore_index=True) if fit_curves else pd.DataFrame()

palette = qualitative.Plotly + qualitative.Dark24 + qualitative.Safe
run_color = {run: palette[i % len(palette)] for i, run in enumerate(run_order)}

fig = go.Figure()
for (run_name, run_label, model, temperature), group in fit_source.groupby(group_cols, dropna=False):
    color = run_color.get(str(run_name), "#636EFA")
    symbol = run_marker.get(str(run_name), "circle")
    fig.add_trace(go.Scatter(
        x=group["k"],
        y=group["pass@k"],
        mode="markers+lines",
        name=f"Observed: {run_label}",
        legendgroup=str(run_name),
        line=dict(color=color, width=1.5),
        marker=dict(color=color, symbol=symbol, size=8),
        customdata=np.stack([group["run_name"], group["model"], group["temperature"]], axis=-1),
        hovertemplate=(
            "run=%{customdata[0]}<br>"
            "model=%{customdata[1]}<br>"
            "temperature=%{customdata[2]}<br>"
            "k=%{x}<br>observed pass@k=%{y:.4f}<extra></extra>"
        ),
    ))

if not linear_fit_curves.empty:
    for (run_name, run_label, model, temperature), group in linear_fit_curves.groupby(group_cols, dropna=False):
        color = run_color.get(str(run_name), "#636EFA")
        fig.add_trace(go.Scatter(
            x=group["k"],
            y=group["fitted_pass@k"],
            mode="lines",
            name=f"Fit to k={EXTRAPOLATE_K_MAX}: {run_label}",
            legendgroup=str(run_name),
            line=dict(color=color, dash="dash", width=3),
            customdata=np.stack([group["run_name"], group["model"], group["temperature"], group["is_extrapolated"]], axis=-1),
            hovertemplate=(
                "run=%{customdata[0]}<br>"
                "model=%{customdata[1]}<br>"
                "temperature=%{customdata[2]}<br>"
                "extrapolated=%{customdata[3]}<br>"
                "k=%{x}<br>fit=%{y:.4f}<extra></extra>"
            ),
        ))

fig.update_layout(
    template="plotly_white",
    title=f"Observed pass@k with linear-regression extrapolation to k={EXTRAPOLATE_K_MAX}",
    xaxis=dict(title="k", range=[1, EXTRAPOLATE_K_MAX], dtick=10),
    yaxis=dict(title="pass@k", range=[0, 1]),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.24,
        xanchor="left",
        x=0,
        tracegroupgap=4,
    ),
    margin=dict(r=30, b=160),
)
fig.show()

linear_fit_params


**This linear model raises classic concerns:**
- Estimates of $\text{pass}_\mathcal{D}@k$ are **not independent** across $k$ when computed from the same samples.
- Estimates are **not homoskedastic**, variance changes over $k$.
- pass@$k$ **may not follow a power law**.

In [ ]:
PLOT_K = max(int(c.split("@")[1]) for c in metric_cols)
temp_df = aggregate.sort_values(["model", "temperature"]).copy()

fig = px.line(
    temp_df,
    x="temperature",
    y=f"pass@{PLOT_K}",
    color="model",
    line_dash="model",
    markers=True,
    title=f"pass@{PLOT_K} by Temperature",
    labels={"temperature": "temperature", f"pass@{PLOT_K}": "pass@k", "model": "Model"},
)
fig.update_traces(line_shape="spline")
fig.update_layout(template="plotly_white", yaxis=dict(range=[0, 1]))
fig.show()


In [ ]:
by_question = pass_at_k[pass_at_k["question_id"] != "__aggregate__"].copy()
by_question.head()


In [ ]:
group_cols = ["model"] + (["temperature"] if "temperature" in samples.columns else [])
agg_spec = {
    "samples": ("correct", "size"),
    "correct": ("correct", "sum"),
    "errors": ("error", lambda s: s.notna().sum()),
    "avg_total_tokens": ("total_tokens", "mean"),
}
if "oracle_raw_response" in samples.columns:
    agg_spec["oracle_calls"] = ("oracle_raw_response", lambda s: s.notna().sum())

sample_stats = samples.groupby(group_cols, dropna=False).agg(**agg_spec).reset_index()
sample_stats
